# consumer_blob.ipynb

Real-Time Food Delivery — **Stream both feeds to Azure Blob Storage as Parquet**

This notebook subscribes to both Event Hubs in parallel and writes the raw
flattened events to Parquet files in your `group10output` Blob container.
This is the "data at rest" layer of the pipeline.

## Architecture

```
group_10_orders   ──┐
                    ├──> Spark Structured Streaming ──┐
group_10_couriers ──┘                                  ├──> Parquet (partitioned)
                                                       │      └─ bronze/orders/
                                                       │      └─ bronze/couriers/
                                                       └──> Checkpoints
                                                              └─ checkpoints/orders/
                                                              └─ checkpoints/couriers/
```

Both writeStreams run in parallel inside the same Spark session. Each has its
own consumer group, checkpoint location, and output path.

## Bronze layer design

- **Path layout:** `bronze/<feed>/event_date=.../zone_id=.../*.parquet`
- **Partitioning:** by `event_date` (for query pruning) and `zone_id` (for zone-level analytics)
- **Trigger:** every 20 seconds — small enough for near-real-time, big enough to avoid tiny files
- **Schema:** flattened from Avro, plus `event_time_ts` (timestamp) and `event_date` (date) added

## Prerequisites
1. Producer notebooks have published events to both hubs.
2. Colab Secrets:
   - `CONSUMER_CONN_STR` — Consumer policy connection string
   - `STORAGE_KEY` — Storage account access key for `iesstsabbadbaa`


## 1. Configuration

In [1]:
from google.colab import userdata

# ─── Azure Event Hub Configuration ──────────────────────────────────────
event_hub_namespace = 'iesstsabbadbaa-grp-06-10'
orders_eventhub     = 'group_10_orders'
couriers_eventhub   = 'group_10_couriers'

# Consumer policy connection string (Listen permission only)
consumer_eventhub_connection_str = userdata.get('CONSUMER_CONN_STR')

# ─── Azure Blob Storage Configuration ───────────────────────────────────
account_name   = 'iesstsabbadbaa'
container_name = 'group10output'

# Storage account access key — store as 'STORAGE_KEY' in Colab Secrets
account_key = userdata.get('STORAGE_KEY')

print(f"Will subscribe to: {orders_eventhub}, {couriers_eventhub}")
print(f"Will write to:     wasbs://{container_name}@{account_name}.blob.core.windows.net/")


Will subscribe to: group_10_orders, group_10_couriers
Will write to:     wasbs://group10output@iesstsabbadbaa.blob.core.windows.net/


In [2]:
!pip install -q fastavro confluent-kafka


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.5/3.5 MB 40.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.0/4.0 MB 101.0 MB/s eta 0:00:00


## 2. Spark Setup

In [3]:
import os
import subprocess

# Fetch the latest Spark 4.x.x version
spark_version = subprocess.run(
    "curl -s https://downloads.apache.org/spark/ | grep -o 'spark-4\\.1\\+\\.[0-9]\\+' | sort -V | tail -1",
    shell=True, capture_output=True, text=True
).stdout.strip()

spark_release = spark_version
hadoop_version = 'hadoop3'

os.environ['SPARK_RELEASE'] = spark_release
os.environ['HADOOP_VERSION'] = hadoop_version
os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-21-openjdk-amd64"
os.environ["SPARK_HOME"] = f"/content/{spark_release}-bin-{hadoop_version}"

print(f"Spark release: {spark_release}")


Spark release: spark-4.1.1


In [4]:
# Install Java 21 and Spark 4.x.y
!apt-get update -qq
!apt-get install openjdk-21-jdk-headless -qq > /dev/null
!wget -q https://downloads.apache.org/spark/${SPARK_RELEASE}/${SPARK_RELEASE}-bin-${HADOOP_VERSION}.tgz
!tar xf ${SPARK_RELEASE}-bin-${HADOOP_VERSION}.tgz

!pip install -q findspark

import findspark
findspark.init()

import pyspark
print("PySpark version:", pyspark.__version__)


W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
PySpark version: 4.1.1


In [5]:
!echo "==== Java JDK: $JAVA_HOME ===="
!/usr/lib/jvm/java-21-openjdk-amd64/bin/java --version
!echo "==== Spark binaries ===="
!ls ${SPARK_RELEASE}-bin-${HADOOP_VERSION}/bin/ | head -5


==== Java JDK: /usr/lib/jvm/java-21-openjdk-amd64 ====
[0.051s][warning][os,container] Cgroup memory controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
[0.051s][warning][os,container] Cgroup cpu controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
openjdk 21.0.10 2026-01-20
OpenJDK Runtime Environment (build 21.0.10+7-Ubuntu-122.04)
OpenJDK 64-Bit Server VM (build 21.0.10+7-Ubuntu-122.04, mixed mode, sharing)
==== Spark binaries ====
beeline
beeline.cmd
docker-image-tool.sh
find-spark-home
find-spark-home.cmd


## 3. Avro Schemas

In [6]:
# Avro schemas as strings — must exactly match producer output
order_schema = """
{
    "type": "record",
    "name": "OrderEvent",
    "namespace": "com.fooddelivery.events.v1",
    "fields": [
        {"name": "event_id",       "type": "string"},
        {"name": "order_id",       "type": "string"},
        {"name": "customer_id",    "type": "string"},
        {"name": "restaurant_id",  "type": "string"},
        {"name": "courier_id",     "type": ["null", "string"], "default": null},
        {"name": "zone_id",        "type": "string"},
        {
            "name": "event_type",
            "type": {
                "type": "enum",
                "name": "OrderEventType",
                "symbols": [
                    "ORDER_PLACED", "ORDER_ACCEPTED", "PREPARING",
                    "READY_FOR_PICKUP", "PICKED_UP", "DELIVERED",
                    "CANCELLED", "REFUND_ISSUED"
                ]
            }
        },
        {"name": "event_time",         "type": "string"},
        {"name": "ingestion_time",     "type": "string"},
        {"name": "order_value",        "type": "float"},
        {"name": "promo_code",         "type": ["null", "string"], "default": null},
        {"name": "promo_discount_pct", "type": ["null", "float"],  "default": null},
        {"name": "estimated_prep_min", "type": "int"},
        {"name": "is_late_event",      "type": "boolean", "default": false},
        {"name": "is_duplicate",       "type": "boolean", "default": false},
        {"name": "schema_version",     "type": "string",  "default": "1.0"}
    ]
}
"""

courier_schema = """
{
    "type": "record",
    "name": "CourierEvent",
    "namespace": "com.fooddelivery.events.v1",
    "fields": [
        {"name": "event_id",    "type": "string"},
        {"name": "courier_id",  "type": "string"},
        {"name": "order_id",    "type": ["null", "string"], "default": null},
        {"name": "zone_id",     "type": "string"},
        {
            "name": "status",
            "type": {
                "type": "enum",
                "name": "CourierStatus",
                "symbols": [
                    "AVAILABLE", "ASSIGNED", "EN_ROUTE_PICKUP",
                    "EN_ROUTE_DELIVERY", "OFFLINE"
                ]
            }
        },
        {"name": "event_time",    "type": "string"},
        {"name": "ingestion_time","type": "string",  "default": ""},
        {"name": "latitude",      "type": "float"},
        {"name": "longitude",     "type": "float"},
        {"name": "speed_kmh",     "type": "float"},
        {"name": "battery_pct",   "type": "int"},
        {"name": "is_late_event", "type": "boolean","default": false},
        {"name": "is_duplicate",  "type": "boolean","default": false},
        {"name": "schema_version","type": "string", "default": "1.0"}
    ]
}
"""

print("Schemas loaded.")


Schemas loaded.


## 4. Spark Session with Azure Blob Storage Access

In [7]:
# JARs required for Spark to talk to:
#   1. Kafka API (Event Hub) for reading
#   2. Avro for deserialization
#   3. Azure Blob Storage (wasbs://) for writing — needs hadoop-azure + azure-storage
jar_dependencies = ",".join([
    "org.apache.spark:spark-sql-kafka-0-10_2.13:4.1.1",
    "org.apache.spark:spark-avro_2.13:4.1.1",
    "org.apache.hadoop:hadoop-azure:3.3.1",
    "com.microsoft.azure:azure-storage:8.6.6",
])

print("JAR coordinates ready.")


JAR coordinates ready.


In [8]:
from pyspark.sql import SparkSession
from pyspark.sql.avro.functions import from_avro

spark = SparkSession \
    .builder \
    .appName("StreamBothFeedsToBlob") \
    .config("spark.streaming.stopGracefullyOnShutdown", True) \
    .config("spark.jars.packages", jar_dependencies) \
    .config(f"fs.azure.account.key.{account_name}.blob.core.windows.net", account_key) \
    .config("spark.sql.shuffle.partitions", 4) \
    .master("local[*]") \
    .getOrCreate()

spark.sparkContext.setLogLevel("WARN")
print("Spark session ready (with Azure Blob auth + Kafka + Avro).")


Spark session ready (with Azure Blob auth + Kafka + Avro).


## 5. Output Paths

In [9]:
# ─── Output paths in Blob Storage ───────────────────────────────────────
# Bronze layer = raw events as received, lightly transformed (flattened, typed).
# We partition by event_date so downstream queries can prune by day.
base = f"wasbs://{container_name}@{account_name}.blob.core.windows.net"

orders_output_path     = f"{base}/bronze/orders/"
orders_checkpoint_path = f"{base}/checkpoints/orders/"

couriers_output_path     = f"{base}/bronze/couriers/"
couriers_checkpoint_path = f"{base}/checkpoints/couriers/"

print("Orders sink:   ", orders_output_path)
print("Couriers sink: ", couriers_output_path)


Orders sink:    wasbs://group10output@iesstsabbadbaa.blob.core.windows.net/bronze/orders/
Couriers sink:  wasbs://group10output@iesstsabbadbaa.blob.core.windows.net/bronze/couriers/


## 6. Kafka Config Helper

In [10]:
def make_kafka_conf(eventhub: str, group_prefix: str) -> dict:
    """Build the Kafka config dict for one Event Hub topic."""
    return {
        "kafka.bootstrap.servers": f"{event_hub_namespace}.servicebus.windows.net:9093",
        "kafka.sasl.mechanism":    "PLAIN",
        "kafka.security.protocol": "SASL_SSL",
        "kafka.sasl.jaas.config":  (
            f'org.apache.kafka.common.security.plain.PlainLoginModule required '
            f'username="$ConnectionString" '
            f'password="{consumer_eventhub_connection_str}";'
        ),
        "subscribe":               eventhub,
        "startingOffsets":         "earliest",
        "groupIdPrefix":           group_prefix,
        "enable.auto.commit":      "true",
        "auto.commit.interval.ms": "5000",
    }

print("Kafka config helper defined.")


Kafka config helper defined.


## 7. Build Streaming DataFrames (one per feed)

In [11]:
# ─── ORDERS stream: read + decode + flatten + add partition columns ──────
from pyspark.sql.functions import col, to_timestamp, to_date

orders_df = spark.readStream \
    .format("kafka") \
    .options(**make_kafka_conf(orders_eventhub, "FoodDelivery_BlobSink_Orders")) \
    .load()

orders_df = orders_df.select(from_avro(orders_df.value, order_schema).alias("o"))

orders_df = orders_df.select(
    col("o.event_id"),
    col("o.order_id"),
    col("o.customer_id"),
    col("o.restaurant_id"),
    col("o.courier_id"),
    col("o.zone_id"),
    col("o.event_type"),
    col("o.event_time"),
    col("o.ingestion_time"),
    col("o.order_value"),
    col("o.promo_code"),
    col("o.promo_discount_pct"),
    col("o.estimated_prep_min"),
    col("o.is_late_event"),
    col("o.is_duplicate"),
    col("o.schema_version"),
) \
.withColumn("event_time_ts", to_timestamp(col("event_time"))) \
.withColumn("event_date",    to_date(col("event_time")))   # for partitioning

orders_df.printSchema()


root
 |-- event_id: string (nullable = true)
 |-- order_id: string (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- restaurant_id: string (nullable = true)
 |-- courier_id: string (nullable = true)
 |-- zone_id: string (nullable = true)
 |-- event_type: string (nullable = true)
 |-- event_time: string (nullable = true)
 |-- ingestion_time: string (nullable = true)
 |-- order_value: float (nullable = true)
 |-- promo_code: string (nullable = true)
 |-- promo_discount_pct: float (nullable = true)
 |-- estimated_prep_min: integer (nullable = true)
 |-- is_late_event: boolean (nullable = true)
 |-- is_duplicate: boolean (nullable = true)
 |-- schema_version: string (nullable = true)
 |-- event_time_ts: timestamp (nullable = true)
 |-- event_date: date (nullable = true)



In [12]:
# ─── COURIERS stream: read + decode + flatten + add partition columns ───
couriers_df = spark.readStream \
    .format("kafka") \
    .options(**make_kafka_conf(couriers_eventhub, "FoodDelivery_BlobSink_Couriers")) \
    .load()

couriers_df = couriers_df.select(from_avro(couriers_df.value, courier_schema).alias("c"))

couriers_df = couriers_df.select(
    col("c.event_id"),
    col("c.courier_id"),
    col("c.order_id"),
    col("c.zone_id"),
    col("c.status"),
    col("c.event_time"),
    col("c.ingestion_time"),
    col("c.latitude"),
    col("c.longitude"),
    col("c.speed_kmh"),
    col("c.battery_pct"),
    col("c.is_late_event"),
    col("c.is_duplicate"),
    col("c.schema_version"),
) \
.withColumn("event_time_ts", to_timestamp(col("event_time"))) \
.withColumn("event_date",    to_date(col("event_time")))

couriers_df.printSchema()


root
 |-- event_id: string (nullable = true)
 |-- courier_id: string (nullable = true)
 |-- order_id: string (nullable = true)
 |-- zone_id: string (nullable = true)
 |-- status: string (nullable = true)
 |-- event_time: string (nullable = true)
 |-- ingestion_time: string (nullable = true)
 |-- latitude: float (nullable = true)
 |-- longitude: float (nullable = true)
 |-- speed_kmh: float (nullable = true)
 |-- battery_pct: integer (nullable = true)
 |-- is_late_event: boolean (nullable = true)
 |-- is_duplicate: boolean (nullable = true)
 |-- schema_version: string (nullable = true)
 |-- event_time_ts: timestamp (nullable = true)
 |-- event_date: date (nullable = true)



## 8. Start Both Sinks

Each `writeStream` runs in the background after `.start()`. Both can run
concurrently within the same Spark session.


In [13]:
# ─── Start the ORDERS → Parquet writeStream ─────────────────────────────
orders_query = orders_df.writeStream \
    .format("parquet") \
    .option("path", orders_output_path) \
    .option("checkpointLocation", orders_checkpoint_path) \
    .partitionBy("event_date", "zone_id") \
    .queryName("orders_to_blob") \
    .trigger(processingTime="20 seconds") \
    .outputMode("append") \
    .start()

print("Started:", orders_query.name, "→", orders_output_path)


Started: orders_to_blob → wasbs://group10output@iesstsabbadbaa.blob.core.windows.net/bronze/orders/


In [14]:
# ─── Start the COURIERS → Parquet writeStream ───────────────────────────
couriers_query = couriers_df.writeStream \
    .format("parquet") \
    .option("path", couriers_output_path) \
    .option("checkpointLocation", couriers_checkpoint_path) \
    .partitionBy("event_date", "zone_id") \
    .queryName("couriers_to_blob") \
    .trigger(processingTime="20 seconds") \
    .outputMode("append") \
    .start()

print("Started:", couriers_query.name, "→", couriers_output_path)


Started: couriers_to_blob → wasbs://group10output@iesstsabbadbaa.blob.core.windows.net/bronze/couriers/


## 9. Monitor Progress

In [15]:
# ─── Monitor both writeStreams ──────────────────────────────────────────
import time

print("Letting both streams run for 60 seconds ...")
time.sleep(60)

for q in spark.streams.active:
    p = q.lastProgress
    if not p:
        print(f"\n{q.name}: no progress yet")
        continue
    print(f"\n{q.name}")
    print(f"  Batch ID:     {p.get('batchId')}")
    print(f"  Input rows:   {p.get('numInputRows')}")
    print(f"  Sink:         {p.get('sink', {}).get('description', 'n/a')}")
    print(f"  Watermark:    {p.get('eventTime', {}).get('watermark', 'n/a')}")


Letting both streams run for 60 seconds ...

couriers_to_blob: no progress yet

orders_to_blob: no progress yet


## 10. Verify Data At Rest

Read the written Parquet files back from Blob Storage and confirm events are
landing correctly. This proves your bronze layer is working end-to-end.


In [16]:
# ─── Verify Parquet files were written for ORDERS ────────────────────────
# Read the written Parquet back and count rows. This proves the data is at
# rest in Blob Storage and queryable by any downstream tool.

try:
    written_orders = spark.read.parquet(orders_output_path)
    print(f"Total order events at rest: {written_orders.count()}")
    print("\nSample rows:")
    written_orders.show(5, truncate=False)

    print("\nEvent counts by zone (from Blob storage):")
    written_orders.groupBy("zone_id").count().orderBy("count", ascending=False).show()
except Exception as e:
    print(f"Couldn't read yet — give the stream more time. Error: {e}")


Total order events at rest: 2385

Sample rows:
+------------------------------------+------------------------------------+-----------+-------------+----------+--------------+------------------------+------------------------+-----------+----------+------------------+------------------+-------------+------------+--------------+-----------------------+----------+-----------+
|event_id                            |order_id                            |customer_id|restaurant_id|courier_id|event_type    |event_time              |ingestion_time          |order_value|promo_code|promo_discount_pct|estimated_prep_min|is_late_event|is_duplicate|schema_version|event_time_ts          |event_date|zone_id    |
+------------------------------------+------------------------------------+-----------+-------------+----------+--------------+------------------------+------------------------+-----------+----------+------------------+------------------+-------------+------------+--------------+-----------------

In [18]:
# ─── Verify Parquet files were written for COURIERS ─────────────────────
try:
    written_couriers = spark.read.parquet(couriers_output_path)
    print(f"Total courier events at rest: {written_couriers.count()}")
    print("\nSample rows:")
    written_couriers.show(5, truncate=False)

    print("\nEvent counts by status (from Blob storage):")
    written_couriers.groupBy("status").count().orderBy("count", ascending=False).show()
except Exception as e:
    print(f"Couldn't read yet — give the stream more time. Error: {e}")


Total courier events at rest: 49918

Sample rows:
+------------------------------------+------------+------------------------------------+---------------+------------------------+------------------------+---------+---------+---------+-----------+-------------+------------+--------------+-------------------+----------+-----------+
|event_id                            |courier_id  |order_id                            |status         |event_time              |ingestion_time          |latitude |longitude|speed_kmh|battery_pct|is_late_event|is_duplicate|schema_version|event_time_ts      |event_date|zone_id    |
+------------------------------------+------------+------------------------------------+---------------+------------------------+------------------------+---------+---------+---------+-----------+-------------+------------+--------------+-------------------+----------+-----------+
|12726ef1-e7eb-4640-88dc-5a2ffa9fc830|courier_0019|NULL                                |AVAILABLE      |

## 11. Inspect Checkpoints (offsets stored in Blob)

In [ ]:
# ─── Inspect Spark's stored Kafka offsets ───────────────────────────────
# Useful to confirm Spark is checkpointing progress so it can resume after
# a restart without losing or duplicating data.

print("=== Orders consumer offsets in Blob ===")
try:
    spark.read.text(f"{orders_checkpoint_path}/offsets/").show(5, truncate=False)
except Exception as e:
    print(f"(no offsets yet: {e})")

print("\n=== Couriers consumer offsets in Blob ===")
try:
    spark.read.text(f"{couriers_checkpoint_path}/offsets/").show(5, truncate=False)
except Exception as e:
    print(f"(no offsets yet: {e})")


## 12. Shutdown

In [ ]:
# Run this when you want to stop the writeStreams cleanly.
if False:  # flip to True to actually stop
    for q in spark.streams.active:
        print(f"Stopping {q.name} ...")
        q.stop()
    spark.stop()
    print("Stopped.")
